## 2. PDF'den Bilgi Çekip Yanıtlama (Document QA)
* Amaç: Öğrencilere veri alma ve RetrievalQA yapısını tanıtmak.
* Senaryo: Öğrenci bir PDF yükler ve belge içeriğine soru sorar.
* Kazandırılan: Belge parçalama, vektörleştirme, arama.
* Kullanılan Bileşenler: PyPDFLoader, FAISS, RetrievalQA

In [ ]:
# ============================================================
# 1) Kurulum
# ============================================================
!pip install -q "langchain>=0.2" "langchain-community>=0.2" "langchain-text-splitters>=0.2" \
               transformers accelerate sentencepiece \
               pypdf faiss-cpu sentence-transformers

# ============================================================
# 2) İçe Aktarımlar
# ============================================================
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

import os, textwrap

# ============================================================
# 3) Model Tanımları (LLM + Embedding)
# ============================================================
# --- LLM: Hafif ve açık kaynak ---
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Alternatif: "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)

gen_pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.2,
    top_p=0.9,
    repetition_penalty=1.1,
)

llm = HuggingFacePipeline(pipeline=gen_pipe)

# --- Embedding: sentence-transformers (hafif ve güçlü) ---
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

# ============================================================
# 4) PDF Yükleme ve Parçalama
# ============================================================
# Kaggle'da dosyayı /kaggle/input/... altına koyabilir veya manuel yükleyebilirsiniz.
PDF_PATH = "/kaggle/input/ozgecmis/CV_2025_industry_academic.pdf"  # <-- kendi PDF yolunuzu girin

loader = PyPDFLoader(PDF_PATH)
docs = loader.load()

# Parçalama: RAG için kısa-örtüşmeli segmentler önerilir.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,        # içerik yoğunluğuna göre 500-1200 arası iyi çalışır
    chunk_overlap=120,     # bağlam taşımak için küçük bir örtüşme
    separators=["\n\n", "\n", " ", ""]
)
chunks = splitter.split_documents(docs)

print(f"Toplam belge sayfası: {len(docs)}, üretilen parça sayısı: {len(chunks)}")

# ============================================================
# 5) Vektör Mağazası (FAISS) Oluşturma
# ============================================================
vectordb = FAISS.from_documents(chunks, embeddings)
retriever = vectordb.as_retriever(search_kwargs={"k": 4})  # en ilgili 4 parçayı getir

# ============================================================
# 6) Sorgu Promptu ve RetrievalQA Zinciri
# ============================================================
# LLM’in RAG çıktısını düzenli ve kaynak-destekli üretmesi için basit bir şablon
prompt_template = PromptTemplate.from_template(
    textwrap.dedent("""
    Aşağıda bir kullanıcı sorusu ve belgeden getirilen ilgili parçalar (context) verilmektedir.
    Sadece bu bağlamdan yararlanarak, akademik ve açık bir Türkçe ile cevap ver.
    Bağlamda yoksa "Metinde bu bilgi yer almıyor." de. Varsayım yapma.

    Soru: {question}

    Bağlam:
    {context}

    Cevap:
    """)
)

# RetrievalQA, varsayılan olarak context ekler; ama çıktıyı kontrol etmek için promptu elle verebiliriz.
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",             # kısa context'ler için "stuff" yeterli
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt_template},
    return_source_documents=True     # kaynak parçaları geri alalım (denetim için)
)

# ============================================================
# 7) Örnek Sorgu
# ============================================================
question = "Belgede anlatılan ana problem ve önerilen çözüm yaklaşımı nedir? Kısa özetler misin?"
result = qa_chain({"query": question})

print("\n⏬ Yanıt:\n")
print(result["result"])

print("\n📚 Kullanılan Kaynak Parçalar (Özet):\n")
for i, d in enumerate(result["source_documents"]):
    print(f"--- Parça {i+1} (s. {d.metadata.get('page', 'NA')}) ---")
    print(textwrap.shorten(d.page_content.replace("\n", " "), width=300, placeholder="..."))
